# Where a thermoelectric cooler rejects its heat

A temperature-controlled sensor is colder than its laboratory surroundings. Increase the current in a thermoelectric module and predict how the available cooling capacity changes. The model treats the effective properties of the entire module as constant. Edit the inputs and rerun the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Tc, Ta = 288., 298.       # cold plate and ambient, K
alpha = .05               # module Seebeck coefficient, V/K
R, K = 2.0, .60           # electrical resistance (ohm), thermal conductance (W/K)
Rs = .45                  # sink-to-air thermal resistance, K/W
selected = 2.5            # operating current, A
assert min(Tc,Ta,alpha,R,K,selected)>0 and Rs >= 0

## Close the hot-side balance before computing cold-side capacity
Use the convective sink relation $T_h=T_a+R_s Q_h$ to eliminate $Q_h$. Heat transport at the hot face contains Peltier heat, half of the electrical dissipation and heat conducted back toward the cold plate. Current is positive in the cooling direction.

In [ ]:
def operating(I, sink):
    denom = 1+sink*(K-alpha*I)
    if np.any(denom <= 0):
        raise ValueError("The chosen hot-side equilibrium is outside the stable constant-property branch.")
    Th = (Ta+sink*(K*Tc + .5*R*I**2))/denom
    Qc = alpha*Tc*I - .5*R*I**2 - K*(Th-Tc)
    volts = R*I + alpha*(Th-Tc)
    Qh = alpha*Th*I + .5*R*I**2 - K*(Th-Tc)
    return Th,Qc,volts,Qh

Th,Qc,V,Qh = operating(selected,Rs)
print(f"Hot face {Th:.2f} K; cold-side capacity {Qc:.2f} W; supply {V:.3f} V; hot-side load {Qh:.2f} W")
print("Cooling COP:", round(Qc/(V*selected),3) if Qc>0 else "no net cooling")
assert np.isclose(Qh-Qc,selected*V)
assert np.isclose(Th-Ta,Rs*Qh)

## Compare the current sweep for two heat sinks
The coefficient of performance is plotted only where the module removes heat and consumes positive electrical power. A passive thermal leak at zero current can therefore appear below zero on the capacity panel.

In [ ]:
amps = np.linspace(0,8.,420)
fig, ax = plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
for resistance,style in [(0.,"--"),(Rs,"-")]:
    t,q,v,qh = operating(amps,resistance)
    power = amps*v
    cop = np.divide(q,power,out=np.full_like(q,np.nan),where=(q>0)&(power>0))
    ax[0].plot(amps,q,style,label=f"Sink {resistance:.2f} K/W")
    ax[1].plot(amps,cop,style,label=f"Sink {resistance:.2f} K/W")
    best = np.argmax(q)
    print(f"In this sweep, sink {resistance:.2f} K/W: capacity peaks near {amps[best]:.2f} A")
ax[0].set(xlabel="Current (A)",ylabel="Heat removed at cold plate (W)",title="Cooling capacity")
ax[1].set(xlabel="Current (A)",ylabel="Cooling COP",title="Cooling per unit electrical input")
for a in ax:
    a.axvline(selected,color="0.5",ls=":",lw=1); a.grid(alpha=.22); a.legend(fontsize=8)
plt.show()

## Decide which operating point serves the measurement
Suppose the sensor dissipates 15 W. Use the arrays to find its lowest feasible drive current on the grid and compare the cooling margin with the point of maximum capacity. Repeat with an ambient temperature 5 K higher. Check whether the hot-face temperature remains within the range where your assumed constant properties are credible.

In [ ]:
target = 15.0
_,capacity,_,_ = operating(amps,Rs)
indices = np.flatnonzero(capacity >= target)
print(f"First current in sweep supplying {target:g} W: {amps[indices[0]]:.3f} A" if len(indices) else "Target unavailable in this sweep")

## Resolve a cooling threshold between sweep points
A sampled current sweep can miss the exact point at which a heat load is first served. Use the rising flank as a bracket for a scalar solve and check the remaining heat-balance error. A local OneSimulation installation supplies its monotone inverse. The browser performs bisection with the same limits.

In [ ]:
if len(indices) and indices[0] > 0:
    first = indices[0]
    lower, upper = amps[first - 1], amps[first]
    def cooling(current):
        return operating(current, Rs)[1]

    try:
        from onesim.numerics.nonlinear.kernels.local_solvers.inversion.bracketed_inverse import bisect_increasing
    except ImportError:
        for _ in range(50):
            centre = (lower + upper) / 2
            if cooling(centre) < target:
                lower = centre
            else:
                upper = centre
        threshold = (lower + upper) / 2
        method = "Notebook bisection"
    else:
        threshold = float(bisect_increasing(cooling, target, lower, upper))
        method = "OneSimulation bracketed inverse"

    print(f"{method}: lower load threshold {threshold:.6f} A")
    assert np.isclose(cooling(threshold), target)
else:
    print("No interior lower crossing for this load and current range.")